In [7]:
%load_ext autoreload
%autoreload 2

from datetime import datetime, timedelta, date
from functools import partial
import numpy as np
import polars as pl

from okx.store import OrderbookStore
from okx.recipes.forwards import build_forwards_pchip, build_forwards_kalman, prepare_pillars
from forwards.pchip import reconstruct_forward, PCHIPCurve
from forwards.kalman_ns import reconstruct_ns_forward, NSCarryState
from forwards.evaluation import wmae_pillar_fit, leave_one_expiry_out, evaluate_curve_snapshot

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
store = OrderbookStore(
    data_root="data/okx",
    manifest_path="data/okx/manifest.sqlite",
    batch_days=5
)


In [9]:
store.clear_cache()

Cleared all caches


In [10]:
# Fix: Convert datetime objects to date objects
dates_fixed = [date(2025, 9, 1)]

pillars = prepare_pillars(
    store,
    inst_family='BTC-USD',
    dates=dates_fixed,
    binning='5m',
    min_time_to_expiry_hours=2.0,
)
print(f"Found {len(pillars)} pillar snapshots")


Found 287 pillar snapshots


In [6]:
if not pillars:
    raise ValueError("No pillar data found for the selected dates and filters.")

for i, (timeMs, (swap, futures)) in enumerate(pillars.items()):
    if i >= 10:
        break
    futures = futures.with_columns(
        pl.col("expiry").cast(pl.Datetime("ms")).alias("expiry_dt")
    )
    print(futures['expiry_dt'].head())



shape: (7,)
Series: 'expiry_dt' [datetime[ms]]
[
	2025-10-31 08:00:00
	2025-12-26 08:00:00
	2025-09-26 07:00:00
	2026-03-27 08:00:00
	2025-09-12 07:00:00
	2025-09-05 07:00:00
	2026-06-26 07:00:00
]
shape: (7,)
Series: 'expiry_dt' [datetime[ms]]
[
	2025-10-31 08:00:00
	2025-12-26 08:00:00
	2025-09-26 07:00:00
	2026-03-27 08:00:00
	2025-09-12 07:00:00
	2025-09-05 07:00:00
	2026-06-26 07:00:00
]
shape: (7,)
Series: 'expiry_dt' [datetime[ms]]
[
	2025-10-31 08:00:00
	2025-12-26 08:00:00
	2025-09-26 07:00:00
	2026-03-27 08:00:00
	2025-09-12 07:00:00
	2025-09-05 07:00:00
	2026-06-26 07:00:00
]
shape: (7,)
Series: 'expiry_dt' [datetime[ms]]
[
	2025-10-31 08:00:00
	2025-12-26 08:00:00
	2025-09-26 07:00:00
	2026-03-27 08:00:00
	2025-09-12 07:00:00
	2025-09-05 07:00:00
	2026-06-26 07:00:00
]
shape: (7,)
Series: 'expiry_dt' [datetime[ms]]
[
	2025-10-31 08:00:00
	2025-12-26 08:00:00
	2025-09-26 07:00:00
	2026-03-27 08:00:00
	2025-09-12 07:00:00
	2025-09-05 07:00:00
	2026-06-26 07:00:00
]
shape: (7,

In [14]:
# Clear cache to pick up the timezone fix
# store.clear_cache()

# Re-fetch pillars with the timezone fix
pillars_fixed = prepare_pillars(
    store,
    inst_family='BTC-USD',
    dates=dates_fixed,
    binning='5m',
    min_time_to_expiry_hours=2.0,
)

print(f"Found {len(pillars_fixed)} pillar snapshots (after timezone fix)")

# Check a few expiry times - should all be 08:00 now
if pillars_fixed:
    first_timeMs, (swap, futures) = next(iter(pillars_fixed.items()))
    futures_check = futures.with_columns(
        pl.col("expiry").cast(pl.Datetime("ms")).alias("expiry_dt")
    )
    print("\nExpiry times (should all be 08:00:00 UTC):")
    print(futures_check.select(pl.col("symbol", "rel_spread", "expiry_dt").head(10)))


Found 287 pillar snapshots (after timezone fix)

Expiry times (should all be 08:00:00 UTC):
shape: (7, 3)
┌───────────────────┬────────────┬─────────────────────┐
│ symbol            ┆ rel_spread ┆ expiry_dt           │
│ ---               ┆ ---        ┆ ---                 │
│ str               ┆ f64        ┆ datetime[ms]        │
╞═══════════════════╪════════════╪═════════════════════╡
│ BTC-USD-260327.OK ┆ 8.8809e-7  ┆ 2026-03-27 08:00:00 │
│ BTC-USD-250912.OK ┆ 0.000114   ┆ 2025-09-12 08:00:00 │
│ BTC-USD-251226.OK ┆ 9.0483e-7  ┆ 2025-12-26 08:00:00 │
│ BTC-USD-250905.OK ┆ 9.2468e-7  ┆ 2025-09-05 08:00:00 │
│ BTC-USD-260626.OK ┆ 0.000037   ┆ 2026-06-26 08:00:00 │
│ BTC-USD-251031.OK ┆ 0.000247   ┆ 2025-10-31 08:00:00 │
│ BTC-USD-250926.OK ┆ 9.2182e-7  ┆ 2025-09-26 08:00:00 │
└───────────────────┴────────────┴─────────────────────┘
